<a href="https://colab.research.google.com/github/rudrapatra23/Brex/blob/main/netflix_user_based.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import kagglehub

# Define the location in Google Drive
DATASET_DIR = "/content/drive/MyDrive/datasets/netflix-prize-data"

# Create the folder if it doesn't exist
os.makedirs(DATASET_DIR, exist_ok=True)

In [ ]:
# Download the dataset directly to Google Drive
if not os.listdir(DATASET_DIR):

    path = kagglehub.dataset_download(
        "netflix-inc/netflix-prize-data",
        output_dir=DATASET_DIR
    )

    print("Dataset downloaded to:", path)

else:
    print("Dataset already exists in Google Drive.")
    print("Using:", DATASET_DIR)

Dataset already exists in Google Drive.
Using: /content/drive/MyDrive/datasets/netflix-prize-data


In [ ]:

file_path = "/content/drive/MyDrive/datasets/netflix-prize-data/combined_data_1.txt"
with open(file_path, "r") as f:
    for i in range(10):
        print(f.readline().strip())

1:
1488844,3,2005-09-06
822109,5,2005-05-13
885013,4,2005-10-19
30878,4,2005-12-26
823519,3,2004-05-03
893988,3,2005-11-17
124105,4,2004-08-05
1248029,3,2004-04-22
1842128,4,2004-05-09


In [ ]:
import pandas as pd

titles_path = "/content/drive/MyDrive/datasets/netflix-prize-data/movie_titles.csv"

movies = []

with open(titles_path, "r", encoding="latin-1") as f:
    for line in f:
        movie_id, year, title = line.strip().split(",", 2)

        movies.append({
            "movie_id": int(movie_id),
            "year": year,
            "title": title
        })

movies = pd.DataFrame(movies)

movies.head()

,movie_id,year,title
0,1,2003,Dinosaur Planet
1,2,2004,Isle of Man TT 2004 Review
2,3,1997,Character
3,4,1994,Paula Abdul's Get Up & Dance
4,5,2004,The Rise and Fall of ECW


In [ ]:
import numpy as np, pandas as pd

# Memory-efficient loader: parse in chunks, store int32/int8 (the list-of-dicts version used GBs of RAM)
def load_ratings(path, chunk_lines=2_000_000):
    parts, cur_movie = [], 0
    for chunk in pd.read_csv(path, header=None, names=["a", "b"], usecols=[0, 1],
                             dtype=str, chunksize=chunk_lines):
        is_movie = chunk["b"].isna()                       # rows like "1:" have no rating
        mid = chunk["a"].where(is_movie).str[:-1].astype("float64")
        mid = mid.ffill().fillna(cur_movie)                # carry movie id forward (also across chunks)
        cur_movie = int(mid.iloc[-1])
        keep = ~is_movie
        parts.append(pd.DataFrame({
            "user_id":  chunk.loc[keep, "a"].astype("int32").values,
            "movie_id": mid[keep].astype("int16").values,
            "rating":   chunk.loc[keep, "b"].astype("int8").values,
        }))
    return pd.concat(parts, ignore_index=True)

ratings_df = load_ratings(os.path.join(DATASET_DIR, "combined_data_1.txt"))
print(ratings_df.shape, ratings_df.memory_usage(deep=True).sum() / 1e6, "MB")

(24053764, 3) 168.37648 MB


In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)

In [ ]:
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix

def build_matrices(train_df):
    user_ids = train_df["user_id"].unique()
    movie_ids = train_df["movie_id"].unique()
    user_to_idx = pd.Series(np.arange(len(user_ids), dtype=np.int32), index=user_ids)
    movie_to_idx = pd.Series(np.arange(len(movie_ids), dtype=np.int32), index=movie_ids)
    rows = user_to_idx.loc[train_df["user_id"]].values
    cols = movie_to_idx.loc[train_df["movie_id"]].values
    vals = train_df["rating"].values.astype(np.float32)
    shape = (len(user_ids), len(movie_ids))
    R = csr_matrix((vals, (rows, cols)), shape=shape)
    cnt = np.diff(R.indptr)
    user_mean = np.asarray(R.sum(axis=1)).ravel() / np.maximum(cnt, 1)
    Rc = R.copy()
    Rc.data = Rc.data - np.repeat(user_mean, cnt).astype(np.float32)
    Rc.data[Rc.data == 0] = 1e-6          # keep explicit entries so the mask stays right
    M = R.copy(); M.data = np.ones_like(M.data)
    Rc_sq = Rc.copy(); Rc_sq.data = Rc_sq.data ** 2
    return dict(R=R, Rc=Rc, M=M, Rc_sq=Rc_sq, user_mean=user_mean.astype(np.float32),
                user_to_idx=user_to_idx, movie_ids=movie_ids, movie_to_idx=movie_to_idx)

def predict_user(mats, u, k=3050100, min_common=15, shrink=51025, min_neighbors=2):
    Rc, M, Rc_sq = mats["Rc"], mats["M"], mats["Rc_sq"]
    rc_u = Rc[u].toarray().ravel(); m_u = M[u].toarray().ravel()
    # 1) intersection size with every user (sparse mat-vec, no n_users x n_users matrix)
    common = M @ m_u
    common[u] = 0
    cands = np.where(common >= min_common)[0]
    if len(cands) == 0: return None
    # 2) similarity computed ONLY over co-rated movies
    num = Rc[cands] @ rc_u
    den_u = np.sqrt(M[cands] @ (rc_u ** 2))
    den_v = np.sqrt(Rc_sq[cands] @ m_u)
    sim = num / np.maximum(den_u * den_v, 1e-9)
    sim *= common[cands] / (common[cands] + shrink)      # trust big overlaps more
    top = np.argsort(-np.abs(sim))[:k]
    nb, s = cands[top], sim[top]
    # 3) weighted average of neighbours' mean-centred ratings
    n_ratings = np.abs(s) @ M[nb]
    score = s @ Rc[nb]
    pred = mats["user_mean"][u] + score / np.maximum(n_ratings, 1e-9)
    n_nb = M[nb].sum(axis=0).A1
    pred[n_nb < min_neighbors] = np.nan
    return np.clip(pred, 1, 5)

def recommend(mats, user_id, movies, n=10, **kw):
    u = mats["user_to_idx"][user_id]
    pred = predict_user(mats, u, **kw)
    if pred is None: return pd.DataFrame()
    seen = mats["M"][u].toarray().ravel() > 0
    pred[seen] = np.nan
    order = np.argsort(-np.nan_to_num(pred, nan=-1))[:n]
    out = pd.DataFrame({"movie_id": mats["movie_ids"][order], "pred_rating": pred[order]})
    return out.merge(movies, on="movie_id", how="left")

In [ ]:
# Example: recommend for a user
movies["movie_id"] = movies["movie_id"].astype(int)
mats = build_matrices(train_df)
sample_user = train_df["user_id"].iloc[0]
recommend(mats, sample_user, movies, n=10)

,movie_id,pred_rating,year,title
0,4238,5.0,2000,Inu-Yasha
1,3115,5.0,2001,Kabhi Khushi Kabhie Gham
2,4427,5.0,2001,The West Wing: Season 3
3,2102,5.0,1994,The Simpsons: Season 6
4,4098,5.0,2002,Angel: Season 4
5,2162,5.0,2000,CSI: Season 1
6,1072,5.0,2000,As Time Goes By: Series 8
7,4383,5.0,2004,Farscape: The Peacekeeper Wars
8,4045,5.0,2003,The Indiana Jones Trilogy: Bonus Material
9,1495,5.0,2001,Alias: Season 1


In [ ]:
# Quick evaluation: RMSE on a sample of users' held-out ratings
import time
rng = np.random.default_rng(0)
test_users = rng.choice(test_df["user_id"].unique(), 200, replace=False)
test_sub = test_df[test_df["user_id"].isin(test_users)]
errs, t0 = [], time.time()
for uid, grp in test_sub.groupby("user_id"):
    if uid not in mats["user_to_idx"].index: continue
    u = mats["user_to_idx"][uid]
    pred = predict_user(mats, u)
    if pred is None: continue
    for mid, r in zip(grp["movie_id"], grp["rating"]):
        j = mats["movie_to_idx"].get(mid)
        if j is not None and not np.isnan(pred[j]):
            errs.append((pred[j] - r) ** 2)
print("RMSE:", np.sqrt(np.mean(errs)), "on", len(errs), "ratings,", round(time.time()-t0,1), "s")

RMSE: 0.9512858 on 1726 ratings, 24.4 s
